# H1 checks: do the worst-ball hypotheses hold across the 2026 death? (Nov 6)

The worst-predicted balls page explains its balls with five hypotheses (A-E). Each was written from a handful of balls. (Nov 7: hypothesis A was split into A, threes, and E, twos, after review.)
This notebook tests each one on **all 3,583 death balls of 2026**: on a slice of balls, how often the outcome happened
minus how often XGBoost predicted it (in percentage points), with a 95% interval from resampling whole matches 2,000 times.

- A positive gap means XGBoost **under-predicted** the outcome on that slice; negative means it over-predicted.
- The hypothesis is supported only if the interval is clear of zero in the direction it predicts.
- "Bonferroni" marks an interval that is also clear at the stricter level for all the rows together.

This is a post-hoc look at 2026: nothing in the models was changed because of it.

Inputs (Drive, read-only): `bundle_2026.parquet` (predictions) and `h1_extract_2026.parquet` (batter and venue history, batting position).
Output (repo, new file): `site/data/h1_checks_2026.csv`.

In [1]:
# ===== H0. Setup (COPY-PASTE) =====
import os
import numpy as np
import pandas as pd

if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/IPL PROJECT ALL DOCS/data/'
    REPO = '/content/ml_death_overs_ipl/'
else:
    DRIVE = os.environ.get('SITE_DRIVE', 'drive/')
    REPO = os.environ.get('SITE_REPO', '../')

b = pd.read_parquet(DRIVE + 'bundle_2026.parquet')
h = pd.read_parquet(DRIVE + 'h1_extract_2026.parquet').set_index('row')
d = b[b['death']].join(h[['bat_six_pct_3s', 'venue_bdry_pct_3s', 'bat_pos']].rename(columns={'bat_pos': 'pos'}))
print('1. death balls:', len(d), '| matches:', d['match_id'].nunique())      # expect 3583 | 71
print('2. missing history values:', int(d[['bat_six_pct_3s', 'venue_bdry_pct_3s', 'pos']].isna().sum().sum()))   # expect 0

1. death balls: 3583 | matches: 71
2. missing history values: 0


In [2]:
# ===== H1. The gap and its match bootstrap (COPY-PASTE) =====
K = {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '6': 5, 'W': 6}     # y codes: index 5 is a six, 6 a wicket

def gap_table(d, rows, n_boot=2000, seed=0, alpha=0.05):
    rng = np.random.default_rng(seed)
    mids = d['match_id'].unique()
    where = {m: np.flatnonzero(d['match_id'].values == m) for m in mids}
    draws = [np.concatenate([where[m] for m in rng.choice(mids, len(mids))]) for _ in range(n_boot)]
    out = []
    for check, label, mask, outs in rows:
        ks = [K[o] for o in outs]
        hit = d['y'].isin(ks).values.astype(float)
        px = d[[f'px_{k}' for k in ks]].sum(axis=1).values
        pb = d[[f'pb1_{k}' for k in ks]].sum(axis=1).values
        mk = mask.values
        full = np.flatnonzero(mk)
        bs = np.array([(hit[ii] - px[ii]).mean() for ii in (dr[mk[dr]] for dr in draws)])
        lo, hi = np.percentile(bs, [2.5, 97.5])
        blo, bhi = np.percentile(bs, [100 * alpha / len(rows) / 2, 100 - 100 * alpha / len(rows) / 2])
        out.append(dict(check=check, slice=label, outcomes='+'.join(outs), balls=len(full),
                        matches=d.loc[mask, 'match_id'].nunique(),
                        actual_pct=100 * hit[full].mean(), xgb_pct=100 * px[full].mean(), b1_pct=100 * pb[full].mean(),
                        gap_pts=100 * (hit[full] - px[full]).mean(), low=100 * lo, high=100 * hi,
                        reading='under-predicted' if lo > 0 else 'over-predicted' if hi < 0 else "can't tell",
                        bonferroni=bool(blo > 0 or bhi < 0)))
    return pd.DataFrame(out)
print('gap_table ready')

gap_table ready


In [3]:
# ===== H2. The five checks (COPY-PASTE) =====
# Slices are fixed by the check questions written on Nov 5, before this was run (A revised Nov 7, after review):
#  A: are threes, on their own, rarer or commoner than XGBoost predicts? (v1 pooled twos and threes for big hitters)
#     (Not split by six rate: top-quarter batters ran 0 threes in 905 balls, and a bootstrap can't give an interval for 0 events.)
#  E: do set big hitters (top quarter for six rate, 20+ balls faced) run more twos than XGBoost predicts?
#  B: do batters in their first three balls, and batters at 8-11, hit more sixes than XGBoost predicts?
#  C: does XGBoost under-predict dots and singles across the 20th over?
#  D: (no check was written) is XGBoost's wicket chance early in the death, with wickets in hand, about right?
six_q = d['bat_six_pct_3s'].quantile(0.75)
ven_q = d['venue_bdry_pct_3s'].quantile(0.75)
big, ground = d['bat_six_pct_3s'] >= six_q, d['venue_bdry_pct_3s'] >= ven_q
new, low_order = d['balls_faced'] <= 2, d['pos'] >= 8
last, early = d['over0'] == 19, (d['over0'] == 15) & (d['wkts_down'] <= 5)
set_in = d['balls_faced'] >= 20
ROWS = [
    ('A', 'All death balls', d['y'] >= 0, ['3']),
    ('B', 'First three balls faced', new, ['6']),
    ('B', 'Batters at No. 8-11', low_order, ['6']),
    ('B', 'First three balls, No. 8-11', new & low_order, ['6']),
    ('B', 'First three balls, No. 1-7', new & ~low_order, ['6']),
    ('C', '20th over', last, ['0']),
    ('C', '20th over', last, ['1']),
    ('C', 'Overs 16-19 (for comparison)', ~last, ['0']),
    ('D', 'Over 16, five or fewer down', early, ['W']),
    ('E', 'Top quarter for six rate, 20+ balls faced', big & set_in, ['2']),
    ('E', 'Top quarter for six rate, any balls faced', big, ['2']),
    ('E', 'Top quarter for six rate, at a top-quarter boundary ground', big & ground, ['2']),
]
res = gap_table(d, ROWS)
print('1. six-rate cut-off (top quarter):', round(six_q, 4), '| venue boundary cut-off:', round(ven_q, 4))
pd.set_option('display.width', 200)
print(res.round(2).to_string(index=False))

1. six-rate cut-off (top quarter): 0.091 | venue boundary cut-off: 0.2309
check                                                      slice outcomes  balls  matches  actual_pct   xgb_pct  b1_pct  gap_pts   low  high         reading  bonferroni
    A                                            All death balls        3   3583       71        0.20  0.160000    0.26     0.03 -0.11  0.21      can't tell       False
    B                                    First three balls faced        6    670       71        5.22  5.540000    7.78    -0.32 -1.94  1.23      can't tell       False
    B                                        Batters at No. 8-11        6    813       56        5.66  7.490000    6.67    -1.83 -3.27 -0.39  over-predicted       False
    B                                First three balls, No. 8-11        6    346       56        4.91  4.390000    6.67     0.53 -1.50  2.64      can't tell       False
    B                                 First three balls, No. 1-7        6    324 

In [4]:
# ===== H3. Save (COPY-PASTE): writes the new file once, then only checks it =====
OUT = REPO + 'site/data/h1_checks_2026.csv'
r = res.round(4)
if os.path.exists(OUT):
    same = pd.read_csv(OUT).round(4).equals(r)
    print('already saved; same numbers:', same)
else:
    r.to_csv(OUT, index=False); print('saved', OUT, r.shape)

saved ../site/data/h1_checks_2026.csv (12, 13)
